In [43]:
#EDA
import pandas as pd
import numpy as np
import plotly.express as px
from plotly._subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from mlxtend.frequent_patterns import apriori, association_rules

### Carregar Dados

In [44]:
df = pd.read_csv('./datasets/transations_by_dept.csv', sep=',')

In [45]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4539 entries, 0 to 4538
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   POS Txn  4539 non-null   uint64
 1   Dept     4539 non-null   str   
 2   ID       4539 non-null   int64 
 3   Sales U  4539 non-null   int64 
dtypes: int64(2), str(1), uint64(1)
memory usage: 142.0 KB


In [46]:
df.head(10)

,POS Txn,Dept,ID,Sales U
0,16120100160021008773,0261:HOSIERY,250,2
1,16120100160021008773,0634:VITAMINS & HLTH AIDS,102,1
2,16120100160021008773,0879:PET SUPPLIES,158,2
3,16120100160021008773,0973:CANDY,175,2
4,16120100160021008773,0982:SPIRITS,176,1
5,16120100160021008773,0983:WINE,177,4
6,16120100160021008773,0991:TOBACCO,179,2
7,16120100160021008774,0597:HEALTH AIDS,93,1
8,16120100160021008774,0604:PERSONAL CARE,100,5
9,16120100160021008775,0819:PRE-RECORDED A/V,135,1


In [47]:
df.tail(10)

,POS Txn,Dept,ID,Sales U
4529,16120100160162000841,0857:PC PERIPH/OFFICE ACC,155,1
4530,16120100160162000841,0931:BATH,165,1
4531,16120100160162000841,0941:BEDDING,167,1
4532,16120100160162000841,0991:TOBACCO,179,1
4533,16120100160162000842,0815:AUDIO ELECTRONICS,132,1
4534,16120100160162000843,0857:PC PERIPH/OFFICE ACC,155,1
4535,16120100160162000845,0395:MENS ATHLETIC SHOES,49,1
4536,16120100160162000845,0815:AUDIO ELECTRONICS,132,1
4537,16120100160162000846,0532:AMERICAN GREETINGS,72,1
4538,16120100160221001467,0066:VENDING/AMUSEMENT MA,242,1


### EDA

In [48]:
df.rename(columns={'POS Txn': 'ID_Transacao', 'Dept': 'Departamento', 'ID': 'ID_Departamento', 'Sales U': 'Qtd_Vendida'}, inplace=True)

In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4539 entries, 0 to 4538
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ID_Transacao     4539 non-null   uint64
 1   Departamento     4539 non-null   str   
 2   ID_Departamento  4539 non-null   int64 
 3   Qtd_Vendida      4539 non-null   int64 
dtypes: int64(2), str(1), uint64(1)
memory usage: 142.0 KB


In [50]:
df['Departamento'].nunique()

160

In [51]:
df['ID_Departamento'].nunique()

160

In [52]:
df['ID_Transacao'].nunique()

2064

In [53]:
len(df.groupby(['ID_Transacao', 'Departamento'])) != len(df)

False

In [54]:
len(df[df['Qtd_Vendida'] <= 0])

149

In [55]:
df = df[df['Qtd_Vendida'] > 0]

In [56]:



fig_volume_transacoes = px.bar(contagem_transacoes_por_departamento, color=contagem_transacoes_por_departamento.index, orientation='h')
fig_volume_transacoes.update_layout(showlegend=False)
fig_volume_transacoes.show()

In [57]:
contagem_transacoes_por_departamento_top10 = df.value_counts('Departamento').head(10)

fig_volume_transacoes_top10 = px.bar(contagem_transacoes_por_departamento_top10, color=contagem_transacoes_por_departamento_top10.index, orientation='h')
fig_volume_transacoes_top10.update_layout(showlegend=False)
fig_volume_transacoes_top10.show()

In [58]:
soma_quantidade_por_departamento = df.groupby('Departamento')['Qtd_Vendida'].sum().sort_values(ascending=False)

fig_unidades_vendidas = px.bar(soma_quantidade_por_departamento, color=soma_quantidade_por_departamento.index, orientation='h')
fig_unidades_vendidas.update_layout(showlegend=False)
fig_unidades_vendidas.show()

In [59]:
soma_quantidade_por_departamento_top10 = df.groupby('Departamento')['Qtd_Vendida'].sum().sort_values(ascending=False).head(10)

fig_unidades_vendidas_top10 = px.bar(soma_quantidade_por_departamento_top10, color=soma_quantidade_por_departamento_top10.index, orientation='h')
fig_unidades_vendidas_top10.update_layout(showlegend=False)
fig_unidades_vendidas_top10.show()

In [60]:
fig_1 = make_subplots(rows=1, cols=2, subplot_titles=('Volume de Transações', 'Unidades Vendidas'))

for trace in fig_volume_transacoes_top10['data']:
    fig_1.add_trace(trace, row=1, col=1)

for trace in fig_unidades_vendidas_top10['data']:
    fig_1.add_trace(trace, row=1, col=2)

fig_1.update_layout(height=800, width=1250, title_text='Top10 Departamentos', showlegend=False)
fig_1.show()

In [61]:
desvio_padrao_quantidade_por_departamento_top20 = df.groupby('Departamento')['Qtd_Vendida'].std().sort_values(ascending=False).head(20)

fig_variacao_unidades_vendidas_top20 = px.bar(desvio_padrao_quantidade_por_departamento_top20, color=desvio_padrao_quantidade_por_departamento_top20.index, orientation='h')
fig_variacao_unidades_vendidas_top20.update_layout(showlegend=False)
fig_variacao_unidades_vendidas_top20.show()

In [62]:
px.box(df, x='Departamento', y='Qtd_Vendida')

In [63]:
media_quantidade_por_departamento_top20 = df.groupby('Departamento')['Qtd_Vendida'].mean().sort_values(ascending=False).head(20)

fig_media_unidades_vendidas_top20 = px.bar(media_quantidade_por_departamento_top20, color=media_quantidade_por_departamento_top20.index, orientation='h')
fig_media_unidades_vendidas_top20.update_layout(showlegend=False)
fig_media_unidades_vendidas_top20.show()

In [64]:
desvio_padrao_quantidade_por_departamento_top20 = df.groupby('Departamento')['Qtd_Vendida'].std().sort_values(ascending=False).head(20)

fig_1 = make_subplots(rows=1, cols=2, subplot_titles=('Desvio Padrão', 'Média'))

for trace in fig_variacao_unidades_vendidas_top20['data']:
    fig_1.add_trace(trace, row=1, col=1)

for trace in fig_media_unidades_vendidas_top20['data']:
    fig_1.add_trace(trace, row=1, col=2)

fig_1.update_layout(height=800, width=1250, title_text='Top20 Departamentos - Unidades Vendidas', showlegend=False)
fig_1.show()